In [17]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

verbose = 0

data_path_list = [
    # "~/workspace/cc2cc_test5/validate/ccdft_atom-1-2008676_g2.csv",
    "~/workspace/cc2cc_test5/validate/ccdft_cc-pVDZ_atom-1-409183_g2.csv",
    "~/workspace/cc2cc_test5/validate/ccdft_cc-pVDZ_atom-1-3177877_g2.csv",
    "~/workspace/cc2cc_test5/validate/ccdft_cc-pVDZ_atom-1-3243248_g2.csv",
    "~/workspace/cc2cc_test5/validate/ccdft_cc-pVDZ_atom-1-3210863_g2.csv",
]
# basis_args = data_path.split("/")[-1].split("_")[1]
basis_args = "cc-pVDZ"
print(basis_args)

with open("../cc2cc/utils/g2.json") as f:
    json_data = json.load(f)

for data_path in data_path_list:
    data = pd.read_csv(data_path)

    data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

    data_name = []
    data_atomic_energy_dft = []
    data_atomic_energy_ai = []
    data_atomic_dft_ele = []
    data_atomic_scf_ele = []
    data_atomic_dft_dip = []
    data_atomic_scf_dip = []

    for i_name in data["name"]:
        if i_name not in json_data["reaction-atomic-energy"]:
            continue
        systems_list = json_data["reaction-atomic-energy"][i_name]["systems"]
        stoichiometry_list = json_data["reaction-atomic-energy"][i_name]["stoichiometry"]

        atomic_energy_dft = 0
        atomic_energy_ai = 0
        for i in range(len(systems_list)):
            atomic_energy_dft += data[data["name"] == systems_list[i]][
                "error_dft_ene"
            ].values[0] * int(stoichiometry_list[i])
            if verbose == 2:
                print(
                    data[data["name"] == systems_list[i]]["error_dft_ene"].values[0],
                    int(stoichiometry_list[i]),
                    systems_list[i],
                )
            atomic_energy_ai += data[data["name"] == systems_list[i]][
                "error_scf_ene"
            ].values[0] * int(stoichiometry_list[i])
        if verbose == 2:
            print(atomic_energy_dft, "\n")
        data_atomic_energy_dft.append(atomic_energy_dft)
        data_atomic_energy_ai.append(atomic_energy_ai)
        data_name.append(i_name)

        data_atomic_dft_ele.append(
            data.loc[data["name"] == i_name, "error_dft_ele"].values[0]
        )
        data_atomic_scf_ele.append(
            data.loc[data["name"] == i_name, "error_scf_ele"].values[0]
        )
        data_atomic_dft_dip.append(
            data.loc[data["name"] == i_name, "error_dft_dip"].values[0]
        )
        data_atomic_scf_dip.append(
            data.loc[data["name"] == i_name, "error_scf_dip"].values[0]
        )

    data_name = np.array(data_name)
    data_atomic_energy_dft = np.array(data_atomic_energy_dft)
    data_atomic_energy_ai = np.array(data_atomic_energy_ai)
    data_atomic_dft_ele = np.array(data_atomic_dft_ele)
    data_atomic_scf_ele = np.array(data_atomic_scf_ele)

    if verbose >= 1:
        sorted_indices = np.argsort(np.abs(data_atomic_energy_dft))[::-1][:10]
        sorted_data_atomic_energy_dft = data_name[sorted_indices]
        print(
            "dft",
            np.array(
                [
                    sorted_data_atomic_energy_dft,
                    np.array(data_atomic_energy_dft)[sorted_indices],
                ]
            ).T,
        )
        sorted_indices = np.argsort(np.abs(data_atomic_energy_ai))[::-1][:10]
        sorted_data_atomic_energy_ai = data_name[sorted_indices]
        print(
            "ai",
            np.array(
                [
                    sorted_data_atomic_energy_ai,
                    np.array(data_atomic_energy_ai)[sorted_indices],
                ]
            ).T,
        )
        print("dft_ele", np.mean(np.abs(data_atomic_dft_ele)))
        print("scf_ele", np.mean(np.abs(data_atomic_scf_ele)))
        print("dft_dip", np.mean(np.abs(data_atomic_dft_dip)))
        print("scf_dip", np.mean(np.abs(data_atomic_scf_dip)))

    print()
    print("Summary of Atomic Energies:")
    print("Mean DFT Energy:", np.mean(np.abs(data_atomic_energy_dft)))
    print("Mean AI Energy:", np.mean(np.abs(data_atomic_energy_ai)))
    print("Mean DFT Ele:", np.mean(np.abs(data_atomic_dft_ele)))
    print("Mean AI Ele:", np.mean(np.abs(data_atomic_scf_ele)))
    print(
        f"Evaluated: {len(data_atomic_energy_dft)}, Total: {len(json_data["reaction-atomic-energy"])}"
    )
    # print("Calculated DFT Energies:", data_atomic_energy_dft)
    # print("Calculated AI Energies:", data_atomic_energy_ai)

    print()

cc-pVDZ

Summary of Atomic Energies:
Mean DFT Energy: 29.834595454794737
Mean AI Energy: 3.9910443517369742
Mean DFT Ele: 0.16224855837276658
Mean AI Ele: 0.12249107112088001
Evaluated: 140, Total: 140


Summary of Atomic Energies:
Mean DFT Energy: 29.8339292016903
Mean AI Energy: 3.5333390391367607
Mean DFT Ele: 0.16224710464352657
Mean AI Ele: 0.16562949898582977
Evaluated: 140, Total: 140


Summary of Atomic Energies:
Mean DFT Energy: 29.834126525893208
Mean AI Energy: 12.028506375534688
Mean DFT Ele: 0.16225042227655045
Mean AI Ele: 0.13346549025720575
Evaluated: 140, Total: 140


Summary of Atomic Energies:
Mean DFT Energy: 7.547810790021498
Mean AI Energy: 1.8696657784786581
Mean DFT Ele: 0.09434726924540233
Mean AI Ele: 0.10333801923790734
Evaluated: 5, Total: 140

